# Chapter 07: Nonholonomic Behavior in Robotic Systems

Source orientation: printed pp. 317-354; PDF pp. 335-372. Chapter question: **When do forbidden instantaneous motions become possible through motion sequences?**

This notebook is an original, standalone teaching chapter. It uses the textbook for structure and mathematical orientation, but the explanations, code, and figures are created here. The chapter focus is Pfaffian distributions, Lie brackets, Frobenius theorem, controllability, Hall bases. The key objects are Pfaffian one-forms, distributions, vector fields, flows, Lie brackets, Frobenius integrability, Chow reachability, growth vectors, and Hall bases.

Generated artifacts live under `artifacts/chapter-07` and are displayed both by code and in the static gallery. The final sanity cell checks the mathematical claims and artifact files so that the notebook remains auditable after reruns.


## Translation Guide

The chapter explains how velocity constraints can still permit finite displacement. A distribution lists the instantaneous directions; Lie brackets reveal directions produced by motion sequences. The computational translation used here is deliberately concrete: every named object becomes an array, graph, cone, trajectory, or map whose dimensions can be checked. That keeps the notebook independent from the PDF while preserving the chapter's mathematical route.

The main objects for this chapter are Pfaffian one-forms, distributions, vector fields, flows, Lie brackets, Frobenius integrability, Chow reachability, growth vectors, and Hall bases. Read each one as a map between spaces. Ask what frame or chart supplies its coordinates, what operation changes those coordinates, and what quantity should remain invariant. The visuals in this notebook are chosen to make those questions inspectable rather than decorative.

The source pages are used only as orientation. Definitions and examples are paraphrased into course language, and all figures are generated from fresh code. When a visual resembles a textbook concept, it is a redraw or computational experiment rather than a page crop.


## Route Through The Chapter

We move in four passes. First, the notebook names the chapter's geometric objects and their engineering purpose. Second, it generates the visual sequence: pfaffian distribution view, lie bracket flow square, growth vector dashboard. Third, it runs a compact computational check that records the relevant residuals, ranks, endpoint errors, determinants, or signs. Fourth, it closes with an applied lab that asks the reader to change a parameter and explain what stayed invariant.

The important habit is to compare the visual artifact with the JSON check. If a cone claims feasibility, the feasibility check should agree. If a Jacobian claims usable motion, the task-space determinant or rank should agree. If a loop claims bracket displacement, the endpoint check should reveal it. The notebook is designed so those cross-checks are near each other.


In [ ]:
from pathlib import Path
import sys
import numpy as np

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the robotics course root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts"
TOPIC = "chapter-07"

from utils.artifacts import display_artifact, save_json
from utils.visuals import build_storyboard, storyboard_check_payload


In [ ]:
storyboard = {
  "label": "Chapter 07: Nonholonomic Behavior in Robotic Systems",
  "artifact_topic": "chapter-07",
  "source_span": "printed pp. 317-354; PDF pp. 335-372",
  "visual_sequence": [
    {
      "kind": "distribution",
      "concept": "Pfaffian Distribution View",
      "filename": "pfaffian-distribution-view.png",
      "observation": "nullspaces of one-forms"
    },
    {
      "kind": "bracket",
      "concept": "Lie Bracket Flow Square",
      "filename": "lie-bracket-flow-square.png",
      "observation": "second-order drift from commutators"
    },
    {
      "kind": "growth",
      "concept": "Growth Vector Dashboard",
      "filename": "growth-vector-dashboard.png",
      "observation": "rank expansion through brackets"
    }
  ]
}

visual_results = build_storyboard(storyboard, ARTIFACT_ROOT, TOPIC)
visual_payload = storyboard_check_payload(storyboard, visual_results)
for item in visual_results:
    display_artifact(item["path"], width=720)
visual_payload


## Static Artifact Gallery

![Pfaffian Distribution View](../../artifacts/chapter-07/figures/pfaffian-distribution-view.png)

*Inspection target:* nullspaces of one-forms.

![Lie Bracket Flow Square](../../artifacts/chapter-07/figures/lie-bracket-flow-square.png)

*Inspection target:* second-order drift from commutators.

![Growth Vector Dashboard](../../artifacts/chapter-07/figures/growth-vector-dashboard.png)

*Inspection target:* rank expansion through brackets.


## Worked Interpretation

The Brockett bracket check symbolically computes a commutator and numerically runs a small flow square. The residual displacement scales with loop area, making the hidden direction visible. This is the chapter's small worked example, not a full simulator. It is intentionally narrow enough that the relevant convention can be seen directly in code and broad enough to catch the common failure mode.

The visual sequence and the executable check use compatible parameters whenever possible. The point is to avoid a split course where the images tell one story and the numbers tell another. When extending this notebook, reuse that pattern: pick one invariant, draw the geometry that exposes it, then save a check payload that would fail if the convention were reversed or the rank condition were lost.

**Pitfall to watch:** Nonholonomic does not mean merely hard to integrate. It means the allowed velocity distribution fails the involutivity test that would make it tangent to configuration-space leaves. This pitfall is why the notebook avoids silent coordinate conventions. Names, frames, dimensions, and signs are part of the teaching surface, not implementation trivia.


In [ ]:
import sympy as sp
from utils.nonholonomic import bracket_loop, lie_bracket_sympy

x, y, z = sp.symbols("x y z")
bracket = lie_bracket_sympy([1, 0, -y], [0, 1, x], [x, y, z])
eps_values = np.array([0.04, 0.06, 0.08])
drifts = np.array([bracket_loop(eps)[2] for eps in eps_values])
fit_order = np.polyfit(np.log(eps_values), np.log(drifts), 1)[0]
check_payload = {
    "lie_bracket": [str(item) for item in bracket],
    "bracket_loop_z_drifts": drifts.round(8).tolist(),
    "drift_order_fit": float(fit_order),
}
assert check_payload["lie_bracket"] == ["0", "0", "2"]
assert 1.8 < check_payload["drift_order_fit"] < 2.2
check_payload


## Applied Lab

Lab prompt: Run a bracket square and fit the drift order as the loop size shrinks.

Before running the lab, make a prediction in three parts: which geometric object is being changed, which representation will show the change most clearly, and which scalar check should move in the same direction. After running it, compare the prediction with the saved JSON payload under `artifacts/chapter-07/labs`.

Use the pitfall above as a diagnostic. If the visual and scalar check disagree, inspect the frame convention, normalization, rank threshold, sign convention, or parameter range. The best result is not merely a green check; it is a short explanation of why the check protects the chapter's main idea.


In [ ]:
lab_notes = {
    "lab": 'Run a bracket square and fit the drift order as the loop size shrinks.',
    "source_orientation": "printed pp. 317-354; PDF pp. 335-372",
    "artifact_topic": TOPIC,
    "reader_task": "Change one parameter, rerun the visual cell, and explain which invariant did or did not change.",
}
lab_path = save_json(lab_notes, TOPIC, "labs", "applied-lab.json", root=ARTIFACT_ROOT)
display_artifact(lab_path)


In [ ]:
from pathlib import Path

visual_paths = [Path(item["path"]) for item in visual_results]
assert visual_payload["assertions"]["has_multiple_visuals"]
assert visual_payload["assertions"]["all_visuals_nonblank"]
assert all(path.exists() and path.stat().st_size > 1000 for path in visual_paths)

final_sanity = {
    "visual_payload": visual_payload,
    "checks": check_payload,
    "visual_artifact_count": len(visual_paths),
    "visual_artifacts": [path.relative_to(BOOK_ROOT).as_posix() for path in visual_paths],
}
sanity_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json", root=ARTIFACT_ROOT)
display_artifact(sanity_path)
final_sanity


## Practice And Inspection Notes

Use this chapter as a small laboratory, not as a static summary. The source span is printed pp. 317-354 and PDF pp. 335-372, but the working material is the notebook itself: the generated artifacts, the executable check payload, and the applied lab. Start by reading the chapter question again: **When do forbidden instantaneous motions become possible through motion sequences?** Then identify which part of the visual sequence gives the most direct answer. For this chapter the focus is Pfaffian distributions, Lie brackets, Frobenius theorem, controllability, Hall bases, so the useful inspection targets are not generic screenshots; they are the ranks, cones, motions, frames, phase loops, energy shapes, or dependency arrows that make that focus concrete.

The `pfaffian distribution view` artifact uses the `distribution` visual family; inspect it for nullspaces of one-forms. The `lie bracket flow square` artifact uses the `bracket` visual family; inspect it for second-order drift from commutators. The `growth vector dashboard` artifact uses the `growth` visual family; inspect it for rank expansion through brackets.

After viewing the artifacts, rerun the computational check cell and read the keys in `check_payload` as a miniature rubric. Each key should protect one concept from a plausible robotics mistake. A determinant or rank protects a singularity claim. A residual protects an equation or closure condition. A monotonicity flag protects a scale-law interpretation. An endpoint error protects a steering construction. A power-invariance error protects a frame transformation. If a check is near zero, ask why zero is the right target; if a check is positive, ask what physical or geometric margin it measures.

Try three variations. First, make a small parameter change that should preserve the chapter's main invariant, such as a coordinate-frame change, a harmless redraw scale, or a sample density increase. Second, make a parameter change that should stress the model, such as a near-singular joint pose, lower friction coefficient, larger controller delay, smaller bracket loop, or weaker tendon tension. Third, make a convention change deliberately, such as reversing a normal or swapping a body/spatial interpretation, and predict which check should fail. These variations turn the notebook from a solved example into a diagnostic tool.

When writing your own notes, use this sentence pattern: "The artifact shows ..., the computation checks ..., and the invariant is ... ." That pattern is intentionally repetitive because it catches vague understanding. A reader who can fill in those three blanks for this chapter can usually transfer the idea to a different mechanism, contact model, or control task without reopening the textbook.


## Takeaways

- A Pfaffian constraint is most useful after plotting its nullspace distribution.
- Lie brackets convert small motion loops into new effective directions.
- Growth vectors summarize how reachability expands with bracket depth.
- Revisit the saved artifacts after changing parameters; the strongest learning usually comes from explaining why the invariant survived.
